# Notebook 15 — Multimodal Integration with Embeddings
Goal: Combine embeddings from different data modalities and use them in a simple predictive workflow.

This notebook demonstrates the common pattern:

`modality A embeddings + modality B features → combined feature space → predictive model`

Multimodal AI integrates information from diverse sources (e.g., images, text, tabular data) to build richer representations. In biology, this could mean combining histology images with genomic data for drug response prediction, where each modality provides complementary signals.

## Why Multimodal Integration?
Single-modality models (e.g., image-only or text-only) can miss important cross-domain relationships. Multimodal integration combines complementary information:

- **Complementary signals**: Images capture spatial patterns (e.g., tissue morphology), while tabular data provides quantitative features (e.g., gene expression levels).
- **Improved robustness**: Combining modalities can reduce overfitting and improve generalization.
- **Biological relevance**: In drug discovery, histology + genomics often outperforms either alone for predicting responses.

This notebook uses simulated data to demonstrate the concept, but the principles apply to real foundation models like CLIP (text-image) or BioCLIP (biology-specific).

## Section 1 — Import Libraries

In [ ]:
# If needed:
# pip install scikit-learn pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## Section 2 — Simulated Image Embeddings
In a real workflow these would come from a vision foundation model such as a Vision Transformer (ViT).

We simulate `n_samples` samples with **16-dimensional embeddings**. Only the **first two dimensions** carry signal correlated with drug response; the remaining 14 are noise. This is a deliberate simplification — real histology embeddings contain far more signal, but the structure (a few informative dims embedded in higher-dimensional noise) is realistic.

In [ ]:
np.random.seed(42)
n_samples = 100

# 16-dimensional embeddings: first 2 dimensions carry tissue morphology signal,
# remaining 14 are noise (irrelevant visual variation).
# 16D is realistic for a small ViT and keeps the noise-to-signal ratio tractable.
image_signal     = np.random.normal(0, 1.0, (n_samples, 2))
image_noise      = np.random.normal(0, 1.0, (n_samples, 14))
image_embeddings = np.hstack([image_signal, image_noise])

image_embeddings.shape  # (100, 16)

## Section 3 — Simulated Molecular Features
A second modality: five genomic/pathway features per sample.

- **KRAS / TP53 / BRAF**: binary mutation indicators (common cancer driver genes)
- **MAPK_score / PI3K_score**: continuous pathway activity scores

These features carry **complementary** signal to the image embeddings — they capture different aspects of biology that the images alone cannot.

In [ ]:
np.random.seed(456)

molecular_features = pd.DataFrame({
    "KRAS_mut":   np.random.binomial(1, 0.3, n_samples),
    "TP53_mut":   np.random.binomial(1, 0.4, n_samples),
    "BRAF_mut":   np.random.binomial(1, 0.1, n_samples),
    "MAPK_score": np.random.normal(0, 0.8, n_samples),
    "PI3K_score": np.random.normal(0, 0.8, n_samples),
})

molecular_features["MAPK_score"] += 0.5 * np.random.normal(0, 0.6, n_samples)
molecular_features["PI3K_score"] += 0.3 * np.random.normal(0, 0.4, n_samples)

molecular_features.head()

## Section 4 — Simulated Clinical Features
This represents patient-level clinical variables that complement molecular and imaging data.

Clinical features add important context:
- **Age**: Patient age can influence treatment response
- **Prior treatment**: Whether patient had previous therapies

These features provide additional predictive signal that's independent of biological measurements.

In [ ]:
np.random.seed(789)

clinical_features = pd.DataFrame({
    "age":             np.random.normal(60, 10, n_samples),
    "prior_treatment": np.random.binomial(1, 0.5, n_samples),
})

clinical_features["age"] += 2 * np.random.normal(0, 3, n_samples)

clinical_features.head()

## Section 5 — Create a Binary Outcome

We construct the outcome so that **image signal and molecular signal each contribute an independent, equal-weight component**. This is the key design choice: neither modality alone can fully predict the outcome, but together they can. Clinical features add a smaller third contribution.

This mirrors real translational settings where histology and genomics capture orthogonal aspects of tumour biology.

In [ ]:
np.random.seed(123)

# Each modality contributes an independent signal component:
#   image_score   — encoded in dims 0 & 1 of the image embeddings only
#   mol_score     — encoded in BRAF/KRAS/MAPK only
#   clinical_score — weak additional contribution
# Neither modality alone can recover the full signal; the combined model can.

image_score    = image_embeddings[:, 0] + image_embeddings[:, 1]

mol_score      = (molecular_features["BRAF_mut"]   * 1.5 +
                  molecular_features["KRAS_mut"]   * 0.8 +
                  molecular_features["MAPK_score"] * 0.6)

clinical_score = (clinical_features["prior_treatment"] * 0.5 +
                  (clinical_features["age"] - 60)       * -0.02)

logits   = image_score + mol_score + clinical_score + np.random.normal(0, 0.8, n_samples)
response = (logits > np.median(logits)).astype(int)   # median split → balanced classes

print(f"Response distribution: {np.bincount(response)} (non-responders, responders)")
print(f"Response rate: {response.mean():.1%}")

## Section 6 — Combine Modalities
We concatenate the embeddings and molecular features into one feature matrix.

## Fusion Strategies
Concatenation is the simplest fusion method: just stack features side-by-side. Other approaches include:

- **Attention-based fusion**: Use transformers to learn how modalities interact (e.g., which image regions relate to gene expression).
- **Cross-modal projection**: Project both modalities to a shared space before combining.
- **Late fusion**: Train separate models per modality and combine predictions.

Here, we use concatenation for simplicity, but advanced methods can capture richer interactions.

In [ ]:
X = np.hstack([image_embeddings, molecular_features.values, clinical_features.values])

X.shape

## Section 7 — Visualise the Combined Feature Space

PCA reduces the 39-dimensional multimodal matrix to 2 dimensions so we can see whether the classes separate.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7, 5))
for label, color, name in [(0, "steelblue", "Non-responder"), (1, "tomato", "Responder")]:
    mask = response == label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, alpha=0.7, s=50)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.title("PCA of Combined Multimodal Feature Space")
plt.legend()
plt.tight_layout()
plt.show()

## Section 8 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, response, test_size=0.3, random_state=42
)

X_train.shape

## Section 9 — Train a Predictive Model

### 9.1 — Multimodal model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f"Multimodal accuracy: {accuracy_score(y_test, preds):.3f}")

### 9.2 — Unimodal baselines

To see whether multimodal fusion helps, compare against models trained on each modality alone.

In [ ]:
# Feature layout in X: [image 0:16 | molecular 16:21 | clinical 21:23]

model_img = LogisticRegression(max_iter=1000)
model_img.fit(X_train[:, :16], y_train)
preds_img = model_img.predict(X_test[:, :16])
print(f"Image-only accuracy:    {accuracy_score(y_test, preds_img):.3f}")

model_mol = LogisticRegression(max_iter=1000)
model_mol.fit(X_train[:, 16:21], y_train)
preds_mol = model_mol.predict(X_test[:, 16:21])
print(f"Molecular-only accuracy: {accuracy_score(y_test, preds_mol):.3f}")

model_clin = LogisticRegression(max_iter=1000)
model_clin.fit(X_train[:, 21:], y_train)
preds_clin = model_clin.predict(X_test[:, 21:])
print(f"Clinical-only accuracy:  {accuracy_score(y_test, preds_clin):.3f}")

## Section 10 — Interpretation
Questions to consider:
- Does combining modalities improve predictive performance over any single modality?
- Which modality contributes the most signal on its own?
- Why might multimodal approaches sometimes *not* help (e.g., too few samples relative to feature count)?
- Would a more powerful model (e.g., gradient boosting) change the ranking?

## Key Takeaways
- Multimodal fusion combines complementary signals from different data types.
- Concatenation is the simplest fusion strategy; attention-based or cross-modal projection methods can capture richer interactions.
- Always compare against unimodal baselines to quantify the value of each modality.
- Having enough samples relative to your feature dimensionality matters — more samples give the model enough data to learn which features are informative.

## Section 11 — Exercises

1. Increase the number of molecular features and retrain the model.
2. Train a model using only image embeddings and compare accuracy.
3. Train a model using only molecular features.
4. Train a model using only clinical features.
5. Change the random seed and observe performance changes.
6. Add additional clinical features (e.g., BMI, smoking status) and retrain.
7. Try different fusion methods (e.g., weighted concatenation instead of simple concatenation).

## Skills Gained
- combining multiple data modalities (images, molecular, clinical)
- building translational multimodal feature matrices
- integrating embeddings with tabular clinical data
- constructing predictive translational pipelines
- evaluating model performance across modalities
- understanding real multimodal AI workflows in biomedicine